In [1]:
pip install google-play-scraper pandas 


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
# Core libraries
import pandas as pd
import numpy as np
import re
from datetime import datetime

# The star of the show
from google_play_scraper import app, reviews, Sort

print("Libraries loaded successfully!")

Libraries loaded successfully!


BOA

In [3]:
# The unique identifier for BOA Bank's app on the Google Play Store
BOA_APP_ID = 'com.boa.boaMobileBanking'

# Step 1: Get app metadata (rating, installs, description...)
app_info1 = app(
    BOA_APP_ID,
    lang='en',    # Language: English
    country='et'  # Country: Ethiopia
)

print("=" * 50)
print("BOA Bank App Info")
print("=" * 50)
print(f"App Title   : {app_info1['title']}")
print(f"Current Score: {app_info1['score']}")
print(f"Total Ratings: {app_info1['ratings']:,}")
print(f"Total Reviews: {app_info1['reviews']:,}")
print(f"Installs     : {app_info1['installs']}")


BOA Bank App Info
App Title   : BoA Mobile
Current Score: 4.3978496
Total Ratings: 9,276
Total Reviews: 1,465
Installs     : 1,000,000+


scraping reviews

In [4]:
# Step 2: Scrape reviews
print(f"Scraping reviews for BOA Bank...")

result1, continuation_token1 = reviews(
    BOA_APP_ID,
    lang='en',
    country='et',
    sort=Sort.NEWEST,       # Most recent first
    count=500,              # Ask for more than 400 to be safe
    filter_score_with=None  # All star ratings
)

print(f"Collected {len(result1)} raw reviews")

Scraping reviews for BOA Bank...
Collected 500 raw reviews


In [5]:
# Let's inspect what a single raw review looks like
print("Keys in a single review:")
print(list(result1[0].keys()))

print("\nFirst raw review (sample):")
for key, value in result1[0].items():
    print(f"  {key}: {value}")

Keys in a single review:
['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']

First raw review (sample):
  reviewId: 9dd3879e-2b0c-4058-8c8c-19f6b376f69c
  userName: Abel Legesse
  userImage: https://play-lh.googleusercontent.com/a-/ALV-UjVlAiPqwEucytjQHmhCbqlROOv44ryrgg5M6NZ2fb1qyojQN2u1
  content: The worst app, also bank am begging for my own money
  score: 1
  thumbsUpCount: 0
  reviewCreatedVersion: 26.05.11
  at: 2026-05-16 12:35:22
  replyContent: None
  repliedAt: None
  appVersion: 26.05.11


In [6]:
# Step 3: Extract only the columns we need
raw_data1 = []

for r in result1:
    raw_data1.append({
        'review_id': r.get('reviewId', ''),
        'review'   : r.get('content', ''),
        'rating'   : r.get('score', None),
        'date'     : r.get('at', None),
        'bank'     : 'BOA Bank',
        'source'   : 'Google Play'
    })

# Build a DataFrame
df_raw1 = pd.DataFrame(raw_data1)

print(f"Shape: {df_raw1.shape}")
df_raw1.head()

Shape: (500, 6)


,review_id,review,rating,date,bank,source
0,9dd3879e-2b0c-4058-8c8c-19f6b376f69c,"The worst app, also bank am begging for my own...",1,2026-05-16 12:35:22,BOA Bank,Google Play
1,5f69466d-ec06-4eb5-816c-296accffeff2,was Good 🙏,5,2026-05-16 00:10:06,BOA Bank,Google Play
2,f9246b8a-6688-4249-b804-0b0c5dd60590,cool,5,2026-05-15 21:07:21,BOA Bank,Google Play
3,21bcff26-5b05-485d-958f-4832ec1fac01,Its Good,5,2026-05-15 15:01:09,BOA Bank,Google Play
4,c5eb7589-59b7-4d72-8aa9-100a703ecaa3,good,5,2026-05-14 21:18:44,BOA Bank,Google Play


Exploring raw data

In [7]:
# Basic shape and types
print(f"Total reviews collected: {len(df_raw1)}")
print(f"\nColumn dtypes:")
print(df_raw1.dtypes)

Total reviews collected: 500

Column dtypes:
review_id               str
review                  str
rating                int64
date         datetime64[us]
bank                    str
source                  str
dtype: object


In [8]:
# Rating distribution — what do users think?
print("Rating distribution:")
rating_counts1 = df_raw1['rating'].value_counts().sort_index(ascending=False)
for rating, count in rating_counts1.items():
    bar = '█' * (count // 5)
    print(f"  {int(rating)} stars: {count:>4}  {bar}")

Rating distribution:
  5 stars:  280  ████████████████████████████████████████████████████████
  4 stars:   37  ███████
  3 stars:   18  ███
  2 stars:   16  ███
  1 stars:  149  █████████████████████████████


In [9]:
# What does the date column look like right now?
print("Sample date values (raw):")
print(df_raw1['date'].head(10).to_string())

print(f"\nDate dtype: {df_raw1['date'].dtype}")

Sample date values (raw):
0   2026-05-16 12:35:22
1   2026-05-16 00:10:06
2   2026-05-15 21:07:21
3   2026-05-15 15:01:09
4   2026-05-14 21:18:44
5   2026-05-12 11:50:32
6   2026-05-11 18:18:54
7   2026-05-09 14:41:50
8   2026-05-08 13:47:07
9   2026-05-07 10:33:06

Date dtype: datetime64[us]


DATA QUALITY AUDIT

In [10]:
print("=" * 50)
print("DATA QUALITY AUDIT")
print("=" * 50)

# --- Problem 1: Missing Values ---
print("\nProblem 1: Missing Values")
print("-" * 30)
missing = df_raw1.isnull().sum()
missing_pct = (missing / len(df_raw1) * 100).round(2)

for col in df_raw1.columns:
    status = f"{missing[col]} missing ({missing_pct[col]}%)" if missing[col] > 0 else "OK"
    print(f"  {col:<15}: {status}")

DATA QUALITY AUDIT

Problem 1: Missing Values
------------------------------
  review_id      : OK
  review         : OK
  rating         : OK
  date           : OK
  bank           : OK
  source         : OK


In [11]:
# --- Problem 2: Duplicate Reviews ---
print("Problem 2: Duplicates")
print("-" * 30)

# Exact duplicates on review text
exact_dupes1 = df_raw1.duplicated(subset=['review']).sum()
print(f"  Exact duplicate reviews : {exact_dupes1}")

# Duplicate review IDs
id_dupes1 = df_raw1.duplicated(subset=['review_id']).sum()
print(f"  Duplicate review IDs    : {id_dupes1}")

# Empty reviews (also a form of bad data)
empty_reviews1 = (df_raw1['review'].str.strip() == '').sum()
print(f"  Empty review texts      : {empty_reviews1}")

Problem 2: Duplicates
------------------------------
  Exact duplicate reviews : 90
  Duplicate review IDs    : 0
  Empty review texts      : 0


In [12]:
# --- Problem 3: Date Format ---
print("Problem 3: Date Format")
print("-" * 30)
print(f"  Current dtype: {df_raw1['date'].dtype}")
print(f"  Sample values: {df_raw1['date'].iloc[0]}")
print(f"  Target format: YYYY-MM-DD (string or date object)")

Problem 3: Date Format
------------------------------
  Current dtype: datetime64[us]
  Sample values: 2026-05-16 12:35:22
  Target format: YYYY-MM-DD (string or date object)


CLEANING STRATEGY

In [13]:
# Work on a copy so raw data stays untouched
df1 = df_raw1.copy()

print(f"Starting with: {len(df1)} reviews")

Starting with: 500 reviews


In [14]:
before1 = len(df1)

# Drop rows missing the critical columns
critical_cols1 = ['review', 'rating']
df = df1.dropna(subset=critical_cols1)

removed = before1 - len(df1)
print(f"Removed {removed} rows with missing critical data")
print(f"Remaining: {len(df1)} reviews")

Removed 0 rows with missing critical data
Remaining: 500 reviews


remove duplIcates

In [15]:
before1 = len(df1)

df1 = df1.drop_duplicates(subset=['review_id'], keep='first')

removed = before1 - len(df1)
print(f"Removed {removed} duplicate reviews")
print(f"Remaining: {len(df1)} reviews")

Removed 0 duplicate reviews
Remaining: 500 reviews


normalizing dates

In [16]:
print("Before normalization:")
print(df1['date'].head(3).to_string())
print(f"dtype: {df1['date'].dtype}")

# Convert to pandas datetime, then format as YYYY-MM-DD string
df1['date'] = pd.to_datetime(df1['date']).dt.strftime('%Y-%m-%d')

print("\nAfter normalization:")
print(df1['date'].head(3).to_string())
print(f"dtype: {df1['date'].dtype}")

print(f"\nDate range: {df1['date'].min()} to {df1['date'].max()}")

Before normalization:
0   2026-05-16 12:35:22
1   2026-05-16 00:10:06
2   2026-05-15 21:07:21
dtype: datetime64[us]

After normalization:
0    2026-05-16
1    2026-05-16
2    2026-05-15
dtype: str

Date range: 2025-02-23 to 2026-05-16
